# Spark Streaming

In [9]:
!pip cache purge --quiet

In [10]:
!pip install install-jdk==1.1.0 \
             nltk==3.9.1 \
             openai==1.99.9 \
             pyspark==4.0.0 --quiet

In [12]:
import jdk
import nltk
import numpy as np
import openai
import os
import pandas as pd
import pyspark
import random
import time
import threading

from nltk.corpus import wordnet as wn
from nltk.tokenize import word_tokenize
from openai import OpenAI
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, input_file_name, to_json, pandas_udf
from pyspark.sql.types import ArrayType, FloatType
from singlestoredb.management import get_secret

os.environ.pop("CONTAINER_ID", None)

'a0f8e711-1866-434e-ab84-a0ab32cfb407'

In [13]:
# Configure Java
jdk_path = jdk.install("17")

os.environ["JAVA_HOME"] = jdk_path
os.environ["PATH"] = f"{jdk_path}/bin:" + os.environ["PATH"]

In [14]:
os.environ["OPENAI_API_KEY"] = get_secret("OPENAI_API_KEY")
openai.api_key = os.environ.get("OPENAI_API_KEY")

In [17]:
# NLTK resources
nltk.download("punkt_tab")
nltk.download("wordnet")

[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [18]:
# Folders
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok = True)

In [19]:
# Define fully-qualified Maven coordinates for JARs
jar_packages = [
    "com.singlestore:singlestore-spark-connector_2.13:4.2.0-spark-4.0.0",
    "com.singlestore:singlestore-jdbc-client:1.2.5",
    "org.apache.commons:commons-dbcp2:2.12.0",
    "org.apache.commons:commons-pool2:2.12.0",
    "io.spray:spray-json_2.13:1.3.6"
]

spark = (
    SparkSession.builder
    .appName("Spark Streaming")
    .master("local[*]")
    .config("spark.executorEnv.OPENAI_API_KEY", openai.api_key)
    .config("spark.driverEnv.OPENAI_API_KEY", openai.api_key)
    .config("spark.jars.packages", ",".join(jar_packages))
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
com.singlestore#singlestore-spark-connector_2.13 added as a dependency
com.singlestore#singlestore-jdbc-client added as a dependency
org.apache.commons#commons-dbcp2 added as a dependency
org.apache.commons#commons-pool2 added as a dependency
io.spray#spray-json_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3c2b3c55-9501-48e5-bb9e-9739cdc6111a;1.0
	confs: [default]
	found com.singlestore#singlestore-spark-connector_2.13;4.2.0-spark-4.0.0 in central
	found org.apache.avro#avro;1.11.3 in central
	found com.fasterxml.jackson.core#jackson-core;2.14.2 in central
	found com.fasterxml.jackson.core#jackson-databind;2.14.2 in central
	found com.fasterxml.jackson.core#jackson-annotat

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select the workspace from the drop-down menu at the top of this notebook.</p>
    </div>
</div>

In [20]:
%%sql
%%sql
CREATE DATABASE IF NOT EXISTS spark_demo_db;

1 rows affected.

++
||
++
++

In [21]:
%%sql
%%sql
USE spark_demo_db;

DROP TABLE IF EXISTS streaming;
CREATE TABLE IF NOT EXISTS streaming (
     value TEXT,
     file_name TEXT,
     embedding VECTOR(1536) NOT NULL
);

++
||
++
++

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select the database from the drop-down menu at the top of this notebook. It updates the <b>connection_url</b> which is used by SQLAlchemy to make connections to the selected database.</p>
    </div>
</div>

In [23]:
from sqlalchemy import *

db_connection = create_engine(connection_url)
url = db_connection.url

In [24]:
password = get_secret("password")
host = url.host
port = url.port
cluster = host + ":" + str(port)

In [25]:
spark.conf.set("spark.datasource.singlestore.ddlEndpoint", cluster)
spark.conf.set("spark.datasource.singlestore.user", "admin")
spark.conf.set("spark.datasource.singlestore.password", password)
spark.conf.set("spark.datasource.singlestore.disablePushdown", "false")

In [26]:
# Producer to simulate new files arriving
def generate_sentence():
    synset = random.choice(list(wn.all_synsets()))
    definition = synset.definition()
    tokens = word_tokenize(definition)
    if tokens:
        tokens[0] = tokens[0].capitalize()
        if not tokens[-1].endswith("."):
            tokens[-1] += "."
    return " ".join(tokens) if tokens else "Placeholder sentence."

def file_producer(dir_path: Path, num_files: int = 20, min_delay = 0, max_delay = 0):
    for i in range(1, num_files + 1):
        time.sleep(random.uniform(min_delay, max_delay))
        fp = dir_path / f"live_file_{i}.txt"
        with fp.open("w") as f:
            f.write(generate_sentence() + "\n")
        print(f"New file created: {fp}")

producer_thread = threading.Thread(target = file_producer, args = (DATA_DIR,), daemon = True)
producer_thread.start()
producer_thread.join()
print("All files created, ready to start Spark streaming")

New file created: data/live_file_1.txt
New file created: data/live_file_2.txt
New file created: data/live_file_3.txt
New file created: data/live_file_4.txt
New file created: data/live_file_5.txt
New file created: data/live_file_6.txt
New file created: data/live_file_7.txt
New file created: data/live_file_8.txt
New file created: data/live_file_9.txt
New file created: data/live_file_10.txt
New file created: data/live_file_11.txt
New file created: data/live_file_12.txt
New file created: data/live_file_13.txt
New file created: data/live_file_14.txt
New file created: data/live_file_15.txt
New file created: data/live_file_16.txt
New file created: data/live_file_17.txt
New file created: data/live_file_18.txt
New file created: data/live_file_19.txt
New file created: data/live_file_20.txt
All files created, ready to start Spark streaming


In [27]:
# Embedding UDF (float32)
@pandas_udf(ArrayType(FloatType()))
def generate_embeddings_batch(texts: pd.Series) -> pd.Series:
    """
    Batch embedding generation for streaming using OpenAI.
    Returns a list of floats (float32) suitable for SingleStore VECTOR.
    """
    client = OpenAI(api_key = os.environ["OPENAI_API_KEY"])
    inputs = texts.fillna("").astype(str).tolist()
    resp = client.embeddings.create(
        model = "text-embedding-3-small",
        input = inputs
    )
    return pd.Series([
        np.array(item.embedding, dtype=np.float32).tolist()
        for item in resp.data
    ])

In [28]:
# Streaming read -> transform -> write
df = (
    spark.readStream
    .format("text")
    .option("path", DATA_DIR)
    .load()
    .withColumn("file_name", input_file_name())
)

# Generate embeddings
df_with_embeddings = df.withColumn("embedding", generate_embeddings_batch("value"))

# Convert embeddings to JSON
df_with_embeddings_json = df_with_embeddings.withColumn(
    "embedding_json",
    to_json(col("embedding"))
)

# Write to SingleStore
def write_to_singlestore(batch_df, batch_id):
    (
        batch_df.select("value", "file_name", "embedding_json")
        .withColumnRenamed("embedding_json", "embedding")
        .write
        .format("singlestore")
        .option("loadDataCompression", "LZ4")
        .option("dbtable", "spark_demo_db.streaming")
        .mode("append")
        .save()
    )
    print(f"Batch {batch_id} written to SingleStore (rows: {batch_df.count()})")

query = (
    df_with_embeddings_json.writeStream
    .foreachBatch(write_to_singlestore)
    .trigger(once = True)
    .start()
)

# Wait for the query to finish processing
while query.isActive:
    time.sleep(1)

INFO:py4j.java_gateway:Callback Server Starting
INFO:py4j.java_gateway:Socket listening on ('127.0.0.1', 40591)
INFO:py4j.clientserver:Python Server ready to receive messages
INFO:py4j.clientserver:Received command c on object id p0


Batch 0 written to SingleStore (rows: 20)


In [29]:
%%sql
%%sql
USE spark_demo_db;

SELECT
    SUBSTR(value, 1, 30) AS value,
    SUBSTR(file_name, LENGTH(file_name) - 9) AS file_name,
    SUBSTR((embedding :> JSON), 1, 30) AS embedding
FROM streaming
LIMIT 5;

5 rows affected.

value,file_name,embedding
Smile affectedly or derisively,ile_10.txt,"[0.0207519531,-0.00992584229,-"
( used of persons ) standing a,ile_17.txt,"[0.0495910645,-0.0401000977,-0"
Refuse to accept and send back,file_1.txt,"[-0.0151824951,0.0413513184,-0"
Of or relating to the United S,file_9.txt,"[0.0133056641,-0.0267791748,0."
Someone who insists on somethi,file_3.txt,"[0.0316772461,-0.0227813721,-0"


In [30]:
spark.stop()